<a href="https://colab.research.google.com/github/vikassinngh123/AI-ML-Learning/blob/main/06-Deep-Learning/01-PyTorch/01-PyTorch-Basic-Models/02_binary_classification_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
import torch
from torch import nn

In [2]:
from sklearn.datasets import load_breast_cancer

cancer=load_breast_cancer()
X,y=cancer['data'],cancer['target']

In [3]:
X[:1]  # X has 30 features

array([[1.799e+01, 1.038e+01, 1.228e+02, 1.001e+03, 1.184e-01, 2.776e-01,
        3.001e-01, 1.471e-01, 2.419e-01, 7.871e-02, 1.095e+00, 9.053e-01,
        8.589e+00, 1.534e+02, 6.399e-03, 4.904e-02, 5.373e-02, 1.587e-02,
        3.003e-02, 6.193e-03, 2.538e+01, 1.733e+01, 1.846e+02, 2.019e+03,
        1.622e-01, 6.656e-01, 7.119e-01, 2.654e-01, 4.601e-01, 1.189e-01]])

In [4]:
y[:20]

array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1])

In [5]:
X.shape , y.shape

((569, 30), (569,))

In [6]:
X=torch.from_numpy(X.astype(np.float32))
y=torch.from_numpy(y.astype(np.float32))

In [7]:
X.dtype , y.dtype

(torch.float32, torch.float32)

In [8]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)

In [9]:
#
device="cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

In [10]:
class cancermodel1(nn.Module):
  def __init__(self):
    super().__init__()

    self.layer1=nn.Linear(in_features=30,out_features=64)    # this layer take 30 feature and upscale it to 64 features
    self.layer2=nn.Linear(in_features=64,out_features=32)    # takes 64 features from the previous layer and downscale it to 32 features
    self.layer3=nn.Linear(in_features=32,out_features=16)    # take 32 features from layer2 and outputs 16 features
    self.layer4=nn.Linear(in_features=16,out_features=1)     # take 16 features from layer3 and outputs a single features

    self.relu=nn.ReLU()
    self.sigmoid=nn.Sigmoid()  #(we are using BCEWithLogitsLoss() that combines a sigmoid layer and BCEloss together)   # convert our ans in a range between 0-1

  def forward(self,x):
    return self.sigmoid(self.layer4(self.relu((self.layer3(self.relu(self.layer2(self.relu(self.layer1(x)))))))))


model_1=cancermodel1().to(device)
model_1

cancermodel1(
  (layer1): Linear(in_features=30, out_features=64, bias=True)
  (layer2): Linear(in_features=64, out_features=32, bias=True)
  (layer3): Linear(in_features=32, out_features=16, bias=True)
  (layer4): Linear(in_features=16, out_features=1, bias=True)
  (relu): ReLU()
  (sigmoid): Sigmoid()
)

In [11]:
loss_fn=nn.BCELoss()
optimizer=torch.optim.SGD(params=model_1.parameters(),lr=0.01)

In [12]:
X_train=X_train.to(device)
y_train=y_train.to(device)
X_test=X_test.to(device)
y_test=y_test.to(device)

In [13]:
epoch=400

for epoch in range(epoch):
  model_1.train()

  y_pred=model_1(X_train)

  loss=loss_fn(y_pred.squeeze(),y_train)

  optimizer.zero_grad()

  loss.backward()

  optimizer.step()

  model_1.eval()

  with torch.inference_mode():
    test_pred=model_1(X_test)
    test_loss=loss_fn(test_pred.squeeze(),y_test)

    predicted_classes = torch.round(test_pred.squeeze())      # round the test_pred to 0 or 1

    correct_predictions = torch.eq(predicted_classes, y_test).sum().item()   # get the number to time our predictions was correct

    total_predictions = len(y_test)     # total lenght of the y_pred for finding accuracy

    accuracy = (correct_predictions / total_predictions) * 100    # Give the model accuracy

  if epoch%100==0 or epoch == 399:
      print(f"epoch:{epoch} accuracy:{accuracy}")

epoch:0 accuracy:62.28070175438597
epoch:100 accuracy:38.59649122807017
epoch:200 accuracy:92.98245614035088
epoch:300 accuracy:94.73684210526315


In [17]:
test_pred.squeeze()[:10]

tensor([0.4747, 0.0167, 0.1579, 0.9159, 0.8885, 0.0911, 0.0194, 0.4305, 0.9214,
        0.9064], device='cuda:0')

In [14]:
print("Predicted:", predicted_classes[:20])
print("Actual:   ", y_test[:20])

Predicted: tensor([0., 0., 0., 1., 1., 0., 0., 0., 1., 1., 1., 0., 1., 1., 1., 1., 1., 1.,
        1., 0.], device='cuda:0')
Actual:    tensor([1., 0., 0., 1., 1., 0., 0., 0., 1., 1., 1., 0., 1., 0., 1., 0., 1., 1.,
        1., 0.], device='cuda:0')


In [15]:
print(f"Test Accuracy: {accuracy:.2f}%")

Test Accuracy: 93.86%
